<center>
<img src="https://drive.google.com/uc?id=1f1gGVI-rxcHjA90WEGNvvtSXF1pAxQwg" alt="Fasilkom UI" width="300"/>

CSGE603130 • Kecerdasan Artifisial dan Sains Data Dasar

Semester Gasal 2024/2025

Fakultas Ilmu Komputer, Universitas Indonesia

**TUGAS KELOMPOK Bing Chilling : *Cycling Segments Leaderboard (CSL)***

<br>
Farras Hafizhudin Indra Wijaya - 2106652682<br>
Arya Daniswara Khairan - 2106702781<br>
I Dewa Putu Aditya Rahman - 2106650456<br>
Ibni Shaquille Syauqi Ibrahim - 2106706735
<center>


## <span style="font-size: 30px; font-weight: bold;">Import Modul yang Dibutuhkan</span>

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.calibration import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from lightgbm import LGBMClassifier

## <span style="font-size: 30px; font-weight: bold;">Mendefine performance evaluation function untuk classifier</span>

In [2]:
def evaluate_classifier_performance(prediction, y_test):
    print("Hasil Evaluasi berdasarkan classification report: \n\n%s\n" % (classification_report(y_test, prediction,zero_division=0)))
    print()
    print("Confusion Matrix")
    y_actual = pd.Series(np.array(y_test), name = "actual")
    y_pred = pd.Series(np.array(prediction), name = "prediction")
    df_confusion = pd.crosstab(y_actual, y_pred)
    display(df_confusion)
    print()

    print('Accuracy Average:', accuracy_score(y_test, prediction))
    print('F1 Macro Average:', f1_score(y_test, prediction, average='macro'))
    print('F1 Micro Average:', f1_score(y_test, prediction, average='micro'))
    print('Precision Macro Average:', precision_score(y_test, prediction, average='macro',zero_division=0))
    print('Precision Micro Average:', precision_score(y_test, prediction, average='micro',zero_division=0))
    print('Recall Macro Average:', recall_score(y_test, prediction, average='macro',zero_division=0))
    print('Recall Micro Average:', recall_score(y_test, prediction, average='micro',zero_division=0))
    print()

## <span style="font-size: 30px; font-weight: bold;">Membaca Dataset, Mengecek outliers, Membuang Duplikat, Memisahkan attempt_date menjadi 3 kolom</span>

In [3]:
df_csl = pd.read_csv("train_csl.csv")
df_csl_copy = df_csl.copy()
df_csl_drop_dupe = df_csl_copy.drop_duplicates()

print(df_csl_drop_dupe.isnull().sum())

ordinal_features = ['user_age_group', 'user_weight_category']
numerical_features = ['smt_rank', 'smt_avg_spd', 'smt_finish_seconds', 
                      'act_avg_spd', 'act_max_spd', 'act_total_km', 'act_moving_seconds', 
                      'act_total_seconds']
categorical_features = ['gender', 'smt_name', 'act_title']

user_age_group            0
user_id                   0
attempt_date              0
gender                    0
smt_rank                  0
smt_avg_spd               0
smt_finish_seconds        0
smt_name                  0
user_weight_category    360
act_title                 0
act_avg_spd               0
act_max_spd               0
act_total_km              0
act_moving_seconds        0
act_total_seconds         0
has_hr_data               0
id                        0
dtype: int64


In [4]:
def count_outliers_iqr(df, columns):
    outlier_counts = {}
    for col in columns:
        if col in df.columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR)))
            outlier_counts[col] = outliers.sum()
        else:
            outlier_counts[col] = 'Column not found'
    
    outlier_counts_df = pd.DataFrame(list(outlier_counts.items()), columns=['Column', 'Outlier Count'])
    outlier_counts_df = outlier_counts_df.set_index('Column')
    return outlier_counts_df

outlier_summary = count_outliers_iqr(df_csl_drop_dupe, numerical_features)

print("Outlier pada tiap atribut numerik:")
print(outlier_summary)

Outlier pada tiap atribut numerik:
                    Outlier Count
Column                           
smt_rank                      163
smt_avg_spd                    53
smt_finish_seconds            288
act_avg_spd                    94
act_max_spd                   375
act_total_km                  308
act_moving_seconds            296
act_total_seconds             296


In [5]:
df_csl_drop_dupe['attempt_date'] = pd.to_datetime(df_csl_drop_dupe['attempt_date'], errors='coerce')
df_csl_drop_dupe['year'] = df_csl_drop_dupe['attempt_date'].dt.year
df_csl_drop_dupe['month'] = df_csl_drop_dupe['attempt_date'].dt.month
df_csl_drop_dupe['day'] = df_csl_drop_dupe['attempt_date'].dt.day

#df_csl_drop_dupe.isnull().sum()
df_csl_drop_dupe.head()

,user_age_group,user_id,attempt_date,gender,smt_rank,smt_avg_spd,smt_finish_seconds,smt_name,user_weight_category,act_title,act_avg_spd,act_max_spd,act_total_km,act_moving_seconds,act_total_seconds,has_hr_data,id,year,month,day
0,25 to 34,1972,2017-12-22,male,712,15.8,382,Headquarters Business Park,54 kg and under,Night Ride,13.6,34.9,19.24,5103,5103,0,6977,2017,12,22
1,25 to 34,239,2015-04-13,male,189,33.2,216,Oghor 2 SailsIsland,105 kg to 114 kg,Night Ride,21.5,43.2,23.47,3926,3926,0,3518,2015,4,13
2,25 to 34,405,2018-02-07,male,264,23.7,593,Starbucks to Majid,75 to 84 kg,Evening Ride,26.7,92.9,38.23,5152,5152,0,415,2018,2,7
3,25 to 34,318,2018-08-24,male,50,33.4,817,Al Fardoos to shellfish round about,75 to 84 kg,Afternoon Ride,31.4,54.4,65.77,7548,7548,1,1755,2018,8,24
4,25 to 34,628,2020-03-06,female,19,21.3,284,Headquarters Business Park,54 kg and under,ثاني تمرين ١٠٠كم,19.9,43.2,96.53,17493,17493,0,7088,2020,3,6


## <span style="font-size: 30px; font-weight: bold;">Lanjutan Preprocessing, Imputasi missing values dengan KNN, dan persiapan training</span>

In [6]:
imputer = KNNImputer(n_neighbors=5)
label_encoder = LabelEncoder()

drop_target = ['gender', 'id', 'user_id', 'attempt_date']

df_csl_drop_dupe[numerical_features] = imputer.fit_transform(df_csl_drop_dupe[numerical_features])

categorical_columns = df_csl_drop_dupe.select_dtypes(include=['object']).columns

df_csl_drop_dupe['gender'] = df_csl_drop_dupe['gender'].map({'female': 1, 'male': 2})

for col in categorical_columns:
    if col != 'gender':
        df_csl_drop_dupe[col] = label_encoder.fit_transform(df_csl_drop_dupe[col])

X_kaggle = df_csl_drop_dupe.drop(drop_target, axis=1)
y_kaggle = df_csl_drop_dupe['gender']

X_train_kaggle, X_test_kaggle, y_train_kaggle, y_test_kaggle = train_test_split(X_kaggle, y_kaggle, train_size=0.80, test_size=0.20, random_state=2025)
X_kaggle.head()

,user_age_group,smt_rank,smt_avg_spd,smt_finish_seconds,smt_name,user_weight_category,act_title,act_avg_spd,act_max_spd,act_total_km,act_moving_seconds,act_total_seconds,has_hr_data,year,month,day
0,2,712.0,15.8,382.0,1,2,505,13.6,34.9,19.24,5103.0,5103.0,0,2017,12,22
1,2,189.0,33.2,216.0,5,0,505,21.5,43.2,23.47,3926.0,3926.0,0,2015,4,13
2,2,264.0,23.7,593.0,8,5,218,26.7,92.9,38.23,5152.0,5152.0,0,2018,2,7
3,2,50.0,33.4,817.0,0,5,127,31.4,54.4,65.77,7548.0,7548.0,1,2018,8,24
4,2,19.0,21.3,284.0,1,2,916,19.9,43.2,96.53,17493.0,17493.0,0,2020,3,6


## <span style="font-size: 30px; font-weight: bold;">Perbandingan 6 model training dengan fungsi evaluate sebelumnya</span>

In [7]:
gnb = GaussianNB()
gnb.fit(X_train_kaggle, y_train_kaggle)
y_pred_kaggle = gnb.predict(X_test_kaggle)

print("Model - Gaussian Naive Bayes")
evaluate_classifier_performance(y_pred_kaggle, y_test_kaggle)

#

random_forest_model = RandomForestClassifier(max_depth=3, n_estimators=50, random_state=2024)
random_forest_model.fit(X_train_kaggle, y_train_kaggle)
y_pred_kaggle = random_forest_model.predict(X_test_kaggle)

print("=====================================================")
print("Model - Random Forest")
evaluate_classifier_performance(y_pred_kaggle, y_test_kaggle)

##

y_train_kaggle_adjusted = y_train_kaggle - 1
y_test_kaggle_adjusted = y_test_kaggle - 1

svm = SVC()
svm.fit(X_train_kaggle, y_train_kaggle_adjusted)

y_pred_kaggle_adjusted = svm.predict(X_test_kaggle)
y_pred_kaggle = y_pred_kaggle_adjusted + 1

print("=====================================================")
print("Model - Support Vector Machine")
evaluate_classifier_performance(y_pred_kaggle, y_test_kaggle)

##

y_train_kaggle_adjusted = y_train_kaggle - 1
y_test_kaggle_adjusted = y_test_kaggle - 1

lgbm = LGBMClassifier()
lgbm.fit(X_train_kaggle, y_train_kaggle_adjusted)

y_pred_kaggle_adjusted = lgbm.predict(X_test_kaggle)
y_pred_kaggle = y_pred_kaggle_adjusted + 1

print("=====================================================")
print("Model - Light Gradient-Boosting Machine")
evaluate_classifier_performance(y_pred_kaggle, y_test_kaggle)

##

y_train_kaggle_adjusted = y_train_kaggle - 1
y_test_kaggle_adjusted = y_test_kaggle - 1

xgboost = XGBClassifier(objective='binary:logistic')
xgboost.fit(X_train_kaggle, y_train_kaggle_adjusted)
y_pred_kaggle_adjusted = xgboost.predict(X_test_kaggle)

y_pred_kaggle = y_pred_kaggle_adjusted + 1

print("=====================================================")
print("Model - Extreme Gradient Boosting")
evaluate_classifier_performance(y_pred_kaggle, y_test_kaggle)

#

catboost = CatBoostClassifier(loss_function='Logloss', eval_metric='F1')
catboost.fit(X_train_kaggle, y_train_kaggle, verbose=False)
y_pred_kaggle = catboost.predict(X_test_kaggle)

print("=====================================================")
print("Model - CatBoost")
evaluate_classifier_performance(y_pred_kaggle, y_test_kaggle)

Model - Gaussian Naive Bayes
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       0.12      0.98      0.22        55
           2       1.00      0.68      0.81      1204

    accuracy                           0.69      1259
   macro avg       0.56      0.83      0.51      1259
weighted avg       0.96      0.69      0.78      1259



Confusion Matrix


prediction,1,2
actual,,
1,54,1
2,390,814



Accuracy Average: 0.6894360603653693
F1 Macro Average: 0.5113863189479504
F1 Micro Average: 0.6894360603653693
Precision Macro Average: 0.5601973138782955
Precision Micro Average: 0.6894360603653693
Recall Macro Average: 0.8289489580187255
Recall Micro Average: 0.6894360603653693

Model - Random Forest
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       1.00      0.02      0.04        55
           2       0.96      1.00      0.98      1204

    accuracy                           0.96      1259
   macro avg       0.98      0.51      0.51      1259
weighted avg       0.96      0.96      0.94      1259



Confusion Matrix


prediction,1,2
actual,,
1,1,54
2,0,1204



Accuracy Average: 0.9571088165210484
F1 Macro Average: 0.50689044911222
F1 Micro Average: 0.9571088165210484
Precision Macro Average: 0.9785373608903021
Precision Micro Average: 0.9571088165210484
Recall Macro Average: 0.509090909090909
Recall Micro Average: 0.9571088165210484

Model - Support Vector Machine
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       0.00      0.00      0.00        55
           2       0.96      1.00      0.98      1204

    accuracy                           0.96      1259
   macro avg       0.48      0.50      0.49      1259
weighted avg       0.91      0.96      0.93      1259



Confusion Matrix


...


Accuracy Average: 0.9952343129467831
F1 Macro Average: 0.9704543321130198
F1 Micro Average: 0.9952343129467831
Precision Macro Average: 0.9881265420075315
Precision Micro Average: 0.9952343129467831
Recall Macro Average: 0.9541301721534279
Recall Micro Average: 0.9952343129467831

Model - Extreme Gradient Boosting
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       0.96      0.93      0.94        55
           2       1.00      1.00      1.00      1204

    accuracy                           1.00      1259
   macro avg       0.98      0.96      0.97      1259
weighted avg       1.00      1.00      1.00      1259



Confusion Matrix


prediction,1,2
actual,,
1,51,4
2,2,1202



Accuracy Average: 0.9952343129467831
F1 Macro Average: 0.9709774089442139
F1 Micro Average: 0.9952343129467831
Precision Macro Average: 0.979473700678995
Precision Micro Average: 0.9952343129467831
Recall Macro Average: 0.9628057988523104
Recall Micro Average: 0.9952343129467831

Model - CatBoost
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       0.98      0.98      0.98        55
           2       1.00      1.00      1.00      1204

    accuracy                           1.00      1259
   macro avg       0.99      0.99      0.99      1259
weighted avg       1.00      1.00      1.00      1259



Confusion Matrix


prediction,1,2
actual,,
1,54,1
2,1,1203



Accuracy Average: 0.9984114376489277
F1 Macro Average: 0.9904938085170643
F1 Micro Average: 0.9984114376489277
Precision Macro Average: 0.9904938085170643
Precision Micro Average: 0.9984114376489277
Recall Macro Average: 0.9904938085170643
Recall Micro Average: 0.9984114376489277



## <span style="font-size: 30px; font-weight: bold;">Preparation untuk data testing, preprocessing & prediction dengan model catboost</span>

In [8]:
df_csl_test = pd.read_csv("test_csl_cla_x.csv")
df_csl_copy = df_csl_test.copy()

df_csl_copy['attempt_date'] = pd.to_datetime(df_csl_copy['attempt_date'], errors='coerce')
df_csl_copy['year'] = df_csl_copy['attempt_date'].dt.year
df_csl_copy['month'] = df_csl_copy['attempt_date'].dt.month
df_csl_copy['day'] = df_csl_copy['attempt_date'].dt.day

df_csl_copy[numerical_features] = imputer.transform(df_csl_copy[numerical_features])

categorical_columns = df_csl_copy.select_dtypes(include=['object']).columns

for col in categorical_columns:
    df_csl_copy[col] = label_encoder.fit_transform(df_csl_copy[col])

X_test_kaggle = df_csl_copy.drop(columns=['id', 'user_id', 'attempt_date'], axis=1)
y_pred_kaggle = catboost.predict(X_test_kaggle)

## <span style="font-size: 30px; font-weight: bold;">Preparation untuk file admission kaggle</span>

In [9]:
kaggle_file_admission = pd.DataFrame({
    "id": df_csl_copy["id"],
    "gender": y_pred_kaggle
})
kaggle_file_admission.to_csv("classification_csl_admission_result_bingchilling.csv", index=False)

## <span style="font-size: 30px; font-weight: bold;">*COMPLEMENTARY* | Testing dengan dataset real</span>

In [10]:
df_og = pd.read_csv('test_csl_cla_x_with_gender.csv')

gender_feat = df_og['gender'].map({'female': 1, 'male': 2})

#

print("Model - Gaussian Naive Bayes")
y_pred_kaggle = gnb.predict(X_test_kaggle)
evaluate_classifier_performance(y_pred_kaggle, gender_feat)

#

print("=====================================================")
print("Model - Random Forest")
y_pred_kaggle = random_forest_model.predict(X_test_kaggle)
evaluate_classifier_performance(y_pred_kaggle, gender_feat)

##

y_pred_kaggle_adjusted = svm.predict(X_test_kaggle)
y_pred_kaggle = y_pred_kaggle_adjusted + 1

print("=====================================================")
print("Model - Support Vector Machine")
evaluate_classifier_performance(y_pred_kaggle, gender_feat)

##

y_pred_kaggle_adjusted = lgbm.predict(X_test_kaggle)
y_pred_kaggle = y_pred_kaggle_adjusted + 1

print("=====================================================")
print("Model - Light Gradient-Boosting Machine")
evaluate_classifier_performance(y_pred_kaggle, gender_feat)

##

print("=====================================================")
print("Model - Extreme Gradient Boosting")
y_pred_kaggle_adjusted = xgboost.predict(X_test_kaggle)
y_pred_kaggle = y_pred_kaggle_adjusted + 1
evaluate_classifier_performance(y_pred_kaggle, gender_feat)

#

print("=====================================================")
print("Model - CatBoost")
y_pred_kaggle = catboost.predict(X_test_kaggle)
evaluate_classifier_performance(y_pred_kaggle, gender_feat)

Model - Gaussian Naive Bayes
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       0.09      0.91      0.17        33
           2       0.99      0.62      0.76       753

    accuracy                           0.63       786
   macro avg       0.54      0.76      0.47       786
weighted avg       0.96      0.63      0.74       786



Confusion Matrix


prediction,1,2
actual,,
1,30,3
2,286,467



Accuracy Average: 0.6323155216284987
F1 Macro Average: 0.4678078003500247
F1 Micro Average: 0.6323155216284987
Precision Macro Average: 0.5442768650686777
Precision Micro Average: 0.6323155216284987
Recall Macro Average: 0.7646384160328383
Recall Micro Average: 0.6323155216284987

Model - Random Forest
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       1.00      0.09      0.17        33
           2       0.96      1.00      0.98       753

    accuracy                           0.96       786
   macro avg       0.98      0.55      0.57       786
weighted avg       0.96      0.96      0.95       786



Confusion Matrix


prediction,1,2
actual,,
1,3,30
2,0,753



Accuracy Average: 0.9618320610687023
F1 Macro Average: 0.5735677083333334
F1 Micro Average: 0.9618320610687023
Precision Macro Average: 0.9808429118773947
Precision Micro Average: 0.9618320610687023
Recall Macro Average: 0.5454545454545454
Recall Micro Average: 0.9618320610687023

Model - Support Vector Machine
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       0.00      0.00      0.00        33
           2       0.96      1.00      0.98       753

    accuracy                           0.96       786
   macro avg       0.48      0.50      0.49       786
weighted avg       0.92      0.96      0.94       786



Confusion Matrix


...


Accuracy Average: 0.9961832061068703
F1 Macro Average: 0.9751964404051878
F1 Micro Average: 0.9961832061068703
Precision Macro Average: 0.998015873015873
Precision Micro Average: 0.9961832061068703
Recall Macro Average: 0.9545454545454546
Recall Micro Average: 0.9961832061068703

Model - Extreme Gradient Boosting
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       1.00      0.88      0.94        33
           2       0.99      1.00      1.00       753

    accuracy                           0.99       786
   macro avg       1.00      0.94      0.97       786
weighted avg       0.99      0.99      0.99       786



Confusion Matrix


prediction,1,2
actual,,
1,29,4
2,0,753



Accuracy Average: 0.9949109414758269
F1 Macro Average: 0.9664174321726127
F1 Micro Average: 0.9949109414758269
Precision Macro Average: 0.9973579920739762
Precision Micro Average: 0.9949109414758269
Recall Macro Average: 0.9393939393939394
Recall Micro Average: 0.9949109414758269

Model - CatBoost
Hasil Evaluasi berdasarkan classification report: 

              precision    recall  f1-score   support

           1       1.00      0.94      0.97        33
           2       1.00      1.00      1.00       753

    accuracy                           1.00       786
   macro avg       1.00      0.97      0.98       786
weighted avg       1.00      1.00      1.00       786



Confusion Matrix


prediction,1,2
actual,,
1,31,2
2,0,753



Accuracy Average: 0.9974554707379135
F1 Macro Average: 0.9837118700265253
F1 Micro Average: 0.9974554707379135
Precision Macro Average: 0.9986754966887417
Precision Micro Average: 0.9974554707379135
Recall Macro Average: 0.9696969696969697
Recall Micro Average: 0.9974554707379135

